# Installation

In [ ]:
!pip install tensorboard

> Can I run tensorboard without a TensorFlow installation?

TensorBoard 1.14+ can be run with a reduced feature set if you do not have TensorFlow installed. The primary limitation is that as of 1.14, only the following plugins are supported: scalars, custom scalars, image, audio, graph, projector (partial), distributions, histograms, text, PR curves, mesh.

# 创建SummaryWriter()

`SummaryWriter()`分为有参和无参的两种形式.

- 无参的是创建在`runs`文件夹下, 按日期再创子文件夹, 同时显示不同的记录.

- 有参的则是创建在指定文件夹下, 只显示最新的记录.
  
  所以, 一般有参的话, 如果是超参数调整时, 应该传入不同的文件夹名`logs-100`和`logs-200`, 而不是都是 `logs`(看不到另一个参数). 然后`tensorboard --logdir=..`(`..`为`logs-100`和`logs-200`的共同父亲目录, 这样就能在一起显示)

接下来的代码都执行两遍, 再打开tensorboard, 以看到区别.

## 无参

默认方式: 存储在当前目录下的默认`runs`目录(没有就自动创建)

In [3]:
from torch.utils.tensorboard import SummaryWriter

# 默认目录 runs
writer = SummaryWriter()

writer.add_scalar('a dot', 200, 0)

writer.close()

In [10]:
%tensorboard --logdir=runs

Reusing TensorBoard on port 6020 (pid 1509942), started 0:21:53 ago. (Use '!kill 1509942' to kill it.)

创建数据后, 点 tensorboard 右上角的刷新按钮, 就能看到数据了.

![picture 2](../image/3add1549e5074d4f9a07e11e3deb8641089082fb1cd64d70599791062664882f.png)  

## 有参

In [5]:
from torch.utils.tensorboard import SummaryWriter

# 写入到当前文件夹下的`logs`文件夹中
writer = SummaryWriter('logs')

writer.add_scalar('a dot', 100, 0)

writer.close()

每次执行会生成一个events.out.tfevents.1667115926.mints.20827.10, tensorboard自动显示的是最新的那个.

![picture 3](../image/9cf548587e653e3c972a16126a3c8b86a433e20f988fd1a1d57ccd6de571fcd5.png)  


In [12]:
%tensorboard --logdir=logs

Reusing TensorBoard on port 6025 (pid 1516019), started 0:00:16 ago. (Use '!kill 1516019' to kill it.)

# 显示tensorboard

`--logdir`保存数据的目录

```bash
.
├── logs
│   ├── events.out.tfevents.1669338915.mints.1512499.2
│   └── events.out.tfevents.1669338918.mints.1512499.3
├── runs
│   ├── Nov25_09-07-44_mints
│   └── Nov25_09-08-17_mints
```
- 要只查看`logs`, 就是`--logdir=logs`

- 要同时显示`logs`和`runs`, 就是`--logdir=..` 
  
  ![picture 4](../image/0b0000caf9961bbeb9285cf8a84a0b4336549927326a7bc11bb23aa08ab4432b.png)  


## jupyter格式

先加载 TensorBoard 的插件

In [8]:
%load_ext tensorboard

再加载 TensorBoard

In [15]:
%tensorboard --logdir=..

Reusing TensorBoard on port 6026 (pid 1517264), started 0:00:05 ago. (Use '!kill 1517264' to kill it.)

## 命令行格式

因为一直运行, 就会阻塞之后的代码, 所以需要在另一个 notebook中执行.

In [ ]:
!tensorboard --logdir=runs

TensorFlow installation not found - running with reduced feature set.

NOTE: Using experimental fast data loading logic. To disable, pass
    "--load_fast=false" and report issues on GitHub. More details:
    https://github.com/tensorflow/tensorboard/issues/4784

Serving TensorBoard on localhost; to expose to the network, use a proxy or pass --bind_all
TensorBoard 2.10.1 at http://localhost:6006/ (Press CTRL+C to quit)
^C


或者在终端中执行`tensorboard --logdir=runs`

# 不同类型的图

## scalar

### add_scalar

In [3]:
from torch.utils.tensorboard import SummaryWriter

# 创建
writer = SummaryWriter()

# x轴
x = range(100)
for i in x:
    # 图的名字, y轴, x轴
    writer.add_scalar('fuction of y', i * 2, i)
    
# 关闭
writer.close()

In [10]:
from torch.utils.tensorboard import SummaryWriter
import numpy as np

writer = SummaryWriter()

for n_iter in range(100):
    writer.add_scalar('Loss/train', np.random.random(), n_iter)
    writer.add_scalar('Loss/test', np.random.random(), n_iter)
    
writer.close()

### add_scalars

In [15]:
from torch.utils.tensorboard import SummaryWriter

writer = SummaryWriter()

r = 5
for i in range(100):
    writer.add_scalars('run_14h',
                       {
                           'xsinx': i*np.sin(i/r),
                           'xcosx': i*np.cos(i/r),
                           'tanx': np.tan(i/r)
                       },
                       i)
writer.close()

## 加图

In [41]:
import torch
# torchvision.datasets.FashionMNIST
import torchvision
# 修改数据集格式
from torchvision import transforms
# data.DataLoader
from torch.utils import data
# nn块
from torch import nn
from torch.utils.tensorboard import SummaryWriter


# 列表
trans = [
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
]
# 转化列表为torchvision.transforms.transforms.Compose对象, 这样就能写 transform=trans
trans = transforms.Compose(trans)
mnist_train_totensor = torchvision.datasets.FashionMNIST(
    root="../data",
    train=True,
    download=True,
    transform=trans
)
mnist_test_totensor = torchvision.datasets.FashionMNIST(
    root="../data",
    train=False,
    download=True,
    transform=trans
)

# shuffle, 打乱
# num_workers, 使用4个进程来读取数据
batch_size = 128
train_iter = data.DataLoader(
    mnist_train_totensor, batch_size, shuffle=True, num_workers=4)
test_iter = data.DataLoader(
    mnist_test_totensor, batch_size, shuffle=True, num_workers=4)

In [42]:
net = torchvision.models.resnet50(False)
# Have ResNet model take in grayscale rather than RGB
net.conv1 = torch.nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
X, y = next(iter(test_iter))

grid = torchvision.utils.make_grid(X)
writer = SummaryWriter()
writer.add_image('images', grid, 0)
writer.add_graph(net, X)
writer.close()

/home/lab/anaconda3/envs/myenv/lib/python3.9/site-packages/torchvision/models/_utils.py:135: UserWarning: Using 'weights' as positional parameter(s) is deprecated since 0.13 and will be removed in 0.15. Please use keyword parameter(s) instead.
  warnings.warn(
/home/lab/anaconda3/envs/myenv/lib/python3.9/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and will be removed in 0.15. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


# 网络例子

In [44]:
# -----------参数-----------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
batch_size = 128
lr = 3e-2
num_epochs=10

cuda


In [45]:
net = nn.Sequential(
    # 这里，我们使用一个11*11的更大窗口来捕捉对象。
    # 同时，步幅为4，以减少输出的高度和宽度。
    # 另外，输出通道的数目远大于LeNet
    nn.Conv2d(1, 96, kernel_size=11, stride=4, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=3, stride=2),
    
    # 减小卷积窗口，使用填充为2来使得输入与输出的高和宽一致，且增大输出通道数
    nn.Conv2d(96, 256, kernel_size=5, padding=2),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=3, stride=2),
    
    # 使用三个连续的卷积层和较小的卷积窗口。
    # 除了最后的卷积层，输出通道的数量进一步增加。
    # 在前两个卷积层之后，汇聚层不用于减少输入的高度和宽度
    nn.Conv2d(256, 384, kernel_size=3, padding=1),
    nn.ReLU(),
    
    nn.Conv2d(384, 384, kernel_size=3, padding=1),
    nn.ReLU(),
    
    nn.Conv2d(384, 256, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=3, stride=2),
    
    nn.Flatten(),
    # 这里，全连接层的输出数量是LeNet中的好几倍。使用dropout层来减轻过拟合
    
    nn.Linear(6400, 4096),
    nn.ReLU(),
    nn.Dropout(p=0.5),
    
    nn.Linear(4096, 4096),
    nn.ReLU(),
    nn.Dropout(p=0.5),
    
    # 最后是输出层。由于这里使用Fashion-MNIST，所以用类别数为10，而非论文中的1000
    nn.Linear(4096, 10)
).to(device)
net

Sequential(
  (0): Conv2d(1, 96, kernel_size=(11, 11), stride=(4, 4), padding=(1, 1))
  (1): ReLU()
  (2): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  (3): Conv2d(96, 256, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
  (4): ReLU()
  (5): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  (6): Conv2d(256, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (7): ReLU()
  (8): Conv2d(384, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (9): ReLU()
  (10): Conv2d(384, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (11): ReLU()
  (12): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  (13): Flatten(start_dim=1, end_dim=-1)
  (14): Linear(in_features=6400, out_features=4096, bias=True)
  (15): ReLU()
  (16): Dropout(p=0.5, inplace=False)
  (17): Linear(in_features=4096, out_features=4096, bias=True)
  (18): ReLU()
  (19): Dropout(p=0.5, inplace=False)
  (20): Linear(in_featu

In [46]:
def init_weights(m):
    if type(m) == nn.Linear or type(m) == nn.Conv2d:
        nn.init.xavier_uniform_(m.weight)


net.apply(init_weights)
optimizer = torch.optim.SGD(net.parameters(), lr=lr)
loss = nn.CrossEntropyLoss()

In [47]:
def train_loop(train_iter, net, loss, optimizer):
    # 共有几批
    num_batchs = len(train_iter)
    # 总平均loss
    total_train_loss = 0
    for batch, (X, y) in enumerate(train_iter):
        # move to device
        X, y = X.to(device), y.to(device)
        # 该批的推断结果
        y_hat = net(X)
        
        train_loss = loss(y_hat, y)
        total_train_loss += train_loss.item()

        # Backpropagation
        optimizer.zero_grad()
        train_loss.backward()
        optimizer.step()

        # --------打印进度        
        print(f"\r[{batch+1:>8d}/{num_batchs:>8d}]  ", end='')

    
    return total_train_loss / num_batchs

加入writer

In [48]:
# ---------训练
writer = SummaryWriter()
for epoch in range(num_epochs):
    total_train_loss = train_loop(train_iter, net, loss, optimizer)
    print(f'epoch {epoch + 1}, total_train_loss {total_train_loss:f}')
    writer.add_scalar('epoch', total_train_loss, epoch+1)

writer.close()

[     469/     469]  epoch 1, total_train_loss 0.969782
[     469/     469]  epoch 2, total_train_loss 0.496370
[     469/     469]  epoch 3, total_train_loss 0.403535
[     469/     469]  epoch 4, total_train_loss 0.358892
[     469/     469]  epoch 5, total_train_loss 0.331865
[     469/     469]  epoch 6, total_train_loss 0.305218
[     469/     469]  epoch 7, total_train_loss 0.287654
[     469/     469]  epoch 8, total_train_loss 0.273580
[     469/     469]  epoch 9, total_train_loss 0.259586
[     469/     469]  epoch 10, total_train_loss 0.246849
